In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

# 1. Authenticate and connect to your workspace
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id="bbeb7561-f822-4950-82f6-64dcae8a93ab",
    resource_group_name="AIMLCC-DEV-RG",
    workspace_name="edu06_histology_img_segmentation"
)

# 2. Specify your pipeline job name (the long GUID/name of the run)
job_name = "helpful_cup_ll7p6ws8mn"

# 3. Download the specific output (e.g., 'annotated_images' or 'clustered_data')
# This downloads the data from Azure Storage to your Compute Instance very quickly
download_path = "./pipeline_outputs"
ml_client.jobs.download(
    name=job_name, 
    download_path=download_path, 
    output_name="final_output" # e.g., "annotated_images"
)

print("Download to Compute Instance complete!")

In [ ]:
import shutil
from pathlib import Path

# 1. Define the path to your final_output folder
# Update this if your download path or output name is different
output_dir = Path("./pipeline_outputs/named-outputs/final_output")

# 2. Iterate through all items in the output directory
for sub_dir in output_dir.iterdir():
    # Check if the item is a directory (the slide sub-folder)
    if sub_dir.is_dir():
        slide_name = sub_dir.name
        
        # Find the JSON file(s) inside this sub-folder
        json_files = list(sub_dir.glob("*.json"))
        
        if json_files:
            # Assuming there is one main JSON file per slide folder
            original_json = json_files[0]
            
            # Define the new path in the root final_output folder, named after the sub-dir
            new_json_path = output_dir / f"{slide_name}.json"
            
            # Move and rename the file
            shutil.move(str(original_json), str(new_json_path))
            print(f"Moved: {slide_name}/{original_json.name} -> {new_json_path.name}")
        
        # 3. Delete the sub-directory (and any leftover hidden files inside it)
        shutil.rmtree(sub_dir)
        print(f"Deleted sub-directory: {slide_name}")

print("\nFile flattening complete!")

In [18]:
import shutil

# Compress the downloaded folder into a single zip file
folder_to_compress = f"{download_path}/named-outputs/final_output"
output_zip_filename = "compressed_pipeline_outputs"

shutil.make_archive(
    base_name=output_zip_filename, 
    format='zip', 
    root_dir=folder_to_compress
)

print(f"Successfully created {output_zip_filename}.zip")

Successfully created compressed_pipeline_outputs.zip


In [ ]:
# Run this in a notebook cell
!zip -r compressed_pipeline_outputs.zip ./pipeline_outputs/named-outputs/final_output

In [ ]:
# The -d flag specifies the destination folder
!unzip "cellpose 4.0.5 annotation jsons.zip" -d "extracted_jsons"

In [ ]:
import shutil
from pathlib import Path

# Define the root folder path
root_folder = Path("pipeline_outputs_Feb_2026_cellpose_4_0_8_cpsam")

# 1. Find all files recursively and move them to the root folder
# rglob("*") finds everything, we filter for files using is_file()
for file_path in root_folder.rglob("*"):
    if file_path.is_file():
        # Skip files that are already in the root folder
        if file_path.parent == root_folder:
            continue
            
        # Define the new destination path in the root folder
        destination_path = root_folder / file_path.name
        
        # Handle potential naming conflicts (if two files have the same name)
        counter = 1
        while destination_path.exists():
            # If a file with the same name exists, append a number to the filename
            destination_path = root_folder / f"{file_path.stem}_{counter}{file_path.suffix}"
            counter += 1
            
        # Move the file
        shutil.move(str(file_path), str(destination_path))
        print(f"Moved: {file_path.name}")

# 2. Clean up empty sub-directories
# We iterate over directories in reverse order (bottom-up) so we delete child folders before parents
for dir_path in sorted(root_folder.rglob("*"), key=lambda p: len(p.parts), reverse=True):
    if dir_path.is_dir() and dir_path != root_folder:
        try:
            # rmdir() only deletes empty directories, which is safe
            dir_path.rmdir()
            print(f"Deleted empty directory: {dir_path.name}")
        except OSError:
            # Directory wasn't empty (maybe hidden files), use rmtree to force delete
            shutil.rmtree(dir_path)
            print(f"Force deleted directory: {dir_path.name}")

print("\nFlattening complete! All files are now in the root folder.")

In [ ]:
import shutil
import os
from pathlib import Path

# 1. Define the folders you want to compress
folder1 = Path("pipeline_outputs_Feb_2026_cellpose_4_0_8_cpsam")
folder2 = Path("pipeline_outputs_July_2025_cellpose_4_0_5_cyto2")

# 2. Create a temporary parent folder to hold both
parent_folder = Path("combined_pipeline_outputs")
parent_folder.mkdir(exist_ok=True)

# 3. Move both folders into the parent folder
# We use shutil.move to move the entire directory structure
if folder1.exists():
    shutil.move(str(folder1), str(parent_folder / folder1.name))
    print(f"Moved {folder1.name} into {parent_folder.name}")
else:
    print(f"Warning: {folder1.name} not found!")

if folder2.exists():
    shutil.move(str(folder2), str(parent_folder / folder2.name))
    print(f"Moved {folder2.name} into {parent_folder.name}")
else:
    print(f"Warning: {folder2.name} not found!")

# 4. Compress the parent folder
output_zip_filename = "combined_pipeline_outputs" # The .zip extension is added automatically

print(f"Compressing {parent_folder.name}...")
shutil.make_archive(
    base_name=output_zip_filename, 
    format='zip', 
    root_dir=str(parent_folder)
)

print(f"Successfully created {output_zip_filename}.zip")